In [ ]:
from SEEQST.tools.setup import generate_experiment, generate_selective_elements, \
    build_parallel_entangler_blocks, build_non_entangling_circuits, parse_circuit, generate_observable_sets
import qutip as qt
import numpy as np
import jax.numpy as jnp
import jax
from SEEQST.tools.processing import process_data,  flatten_list, parse_circuit_to_qobj, \
    data_predict_from_rho_sampled, prepare_state, density_matrix


In [ ]:
# Experiment setup
n_qubits = 10 
N_shadow = 100
# sample N settings, i.e. blocks for shadow
seed = 1234
key = jax.random.PRNGKey(seed)
block_indices = jax.random.randint(key, (N_shadow,),minval=0, maxval=2**n_qubits).tolist()
# result = generate_selective_elements(block_indices, None, n_qubits)
observable_dict = generate_observable_sets(block_indices, n_qubits)
sel_circ_text = build_parallel_entangler_blocks(block_indices, n_qubits)

In [ ]:
# ignore, there is a rand already
def random_state(n_qubits, key, args: dict, rand_generator):
    shape = (2**n_qubits, 2**n_qubits)
    key1, key2 = jax.random.split(key)
    T = rand_generator(key=key1, shape=shape, **args) + 1j*rand_generator(key=key2, shape=shape, **args) 
    return density_matrix(T)

In [13]:
X = jnp.array([[0,1],
              [1,0]], dtype=jnp.complex128)

Y = jnp.array([[0,-1j],
              [1j,0]], dtype=jnp.complex128)

Z = jnp.array([[1,0],
              [0,-1]], dtype=jnp.complex128)

I = jnp.array([[1,0],
              [0,1]], dtype=jnp.complex128)

def observable_from_string(obs_string):
    obs_dict = {"I": I, "X": X, "Y": Y, "Z": Z}
    res = jnp.array([1], dtype=jnp.complex128)
    for char in obs_string:
        res = jnp.kron(res, obs_dict[char])

    return res

/var/folders/6_/cjrk611x35z8hr44jyg5dkfm0000gn/T/ipykernel_45226/1197604039.py:1: UserWarning: Explicitly requested dtype <class 'jax.numpy.complex128'> requested in array is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  X = jnp.array([[0,1],
/var/folders/6_/cjrk611x35z8hr44jyg5dkfm0000gn/T/ipykernel_45226/1197604039.py:4: UserWarning: Explicitly requested dtype <class 'jax.numpy.complex128'> requested in array is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  Y = jnp.array([[0,-1j],
/var/folders/6_/cjrk611x35z8hr44jyg5dkfm0000gn/T/ipykernel_45226/1197604039.py:7: UserWarning: Explicitly requested dtype <class 'ja

In [ ]:
# Convert circuit text to JAX arrays
# Selective circuit texts - can be modified to improve efficiency
circuits = flatten_list(sel_circ_text)  # Requires CNOT gates

# circuits = flatten_list(sel_circ_text_non_entangle) # Uncomment to use non-entangling circuits

# Convert circuit text to unitary matrices
unitaries = parse_circuit_to_qobj(circuits, n_qubits)

# Convert Qobj to JAX numpy arrays
unitaries_jnp = jnp.array([uni.full() for uni in unitaries])
rho_gt = qt.rand_dm([[2]*n_qubits])  # Random density matrix
data = (data_predict_from_rho_sampled(jnp.array(rho_gt.full()), unitaries_jnp, shots=1))


KeyboardInterrupt: 

In [ ]:
# Process the data to reconstruct the density matrix
rho_hat=process_data(data=data,unitaries_jnp=unitaries_jnp,selective_blocks=block_indices,shots=1,N=n_qubits)
obs_string = ["Z" for _ in range(n_qubits)] # can also change this
obs = observable_from_string(obs_string)
shadow_estimate = jnp.trace(rho_hat @ obs)
gt_property = jnp.trace(rho_gt @ obs)
print("Ground truth: ", gt_property, " prediction: ", shadow_estimate, " Error: ", jnp.abs(gt_property-shadow_estimate))
